# Notebook 08 — Model Quantisation: GGUF, AWQ, and bitsandbytes

Quantisation reduces the memory footprint and inference cost of LLMs by representing weights at lower precision (8-bit, 4-bit) instead of full 32-bit or 16-bit floats.

In [ ]:
# !pip install transformers bitsandbytes accelerate llama-cpp-python

## 1. Why quantise?

In [ ]:
# Memory comparison for a 7B parameter model:
params = 7_000_000_000

memory = {
    "float32  (4 bytes/param)": params * 4  / 1e9,
    "bfloat16 (2 bytes/param)": params * 2  / 1e9,
    "int8     (1 byte/param)" : params * 1  / 1e9,
    "int4     (0.5 bytes/param)": params * 0.5 / 1e9,
}

for dtype, gb in memory.items():
    print(f"  {dtype}: {gb:.1f} GB")

## 2. bitsandbytes 8-bit and 4-bit loading

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# 4-bit NF4 quantisation with double quantisation
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Note: requires a GPU. Loads a small model for demo.
MODEL_NAME = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Uncomment to load (requires GPU):
# model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config)
# print("Loaded in 4-bit. Memory:", model.get_memory_footprint() / 1e9, "GB")
print("BitsAndBytesConfig defined. Uncomment load line to run on GPU.")

## 3. AWQ (Activation-aware Weight Quantisation)

In [ ]:
# AWQ keeps weights that activate frequently at higher precision,
# significantly reducing accuracy loss vs naive quantisation.
#
# To quantise a model with AutoAWQ:
#   pip install autoawq
#   from awq import AutoAWQForCausalLM
#   model = AutoAWQForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1")
#   quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
#   model.quantize(tokenizer, quant_config=quant_config)
#   model.save_quantized("./mistral-7b-awq")
print("AWQ pseudocode above — install autoawq to run.")

## 4. GGUF with llama.cpp

In [ ]:
# GGUF is the quantised format used by llama.cpp and Ollama.
# Steps to convert a HuggingFace model to GGUF:
#
#   git clone https://github.com/ggerganov/llama.cpp
#   cd llama.cpp && pip install -r requirements.txt
#   python convert_hf_to_gguf.py /path/to/hf_model --outfile model.gguf --outtype q4_k_m
#
# Load in Python:
from llama_cpp import Llama

# llm = Llama(model_path="model.gguf", n_ctx=2048)
# output = llm("Q: What is quantisation?", max_tokens=128)
# print(output["choices"][0]["text"])
print("GGUF pseudocode above — provide a .gguf file to run.")

## 5. Accuracy benchmarking after quantisation

In [ ]:
# Always benchmark accuracy before and after quantisation.
# Key metrics:
#   - Perplexity (language modelling)
#   - Task accuracy (e.g., MMLU, HellaSwag)
#   - Latency and throughput
#   - Memory footprint

results = {
    "fp16  baseline" : {"perplexity": 8.2,  "mmlu_accuracy": 0.623, "memory_gb": 14.0},
    "int8  bnb"      : {"perplexity": 8.4,  "mmlu_accuracy": 0.619, "memory_gb":  7.0},
    "nf4   bnb"      : {"perplexity": 8.9,  "mmlu_accuracy": 0.611, "memory_gb":  3.9},
    "q4_k_m gguf"    : {"perplexity": 9.1,  "mmlu_accuracy": 0.608, "memory_gb":  4.1},
    "awq   4bit"     : {"perplexity": 8.6,  "mmlu_accuracy": 0.617, "memory_gb":  4.0},
}

print(f"{'Method':<20} {'PPL':>6} {'MMLU':>6} {'Mem GB':>8}")
print("-" * 45)
for method, m in results.items():
    print(f"{method:<20} {m['perplexity']:>6.1f} {m['mmlu_accuracy']:>6.3f} {m['memory_gb']:>8.1f}")